In [1]:
import os
import sys

import math
import time
import datetime
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import torch
from torch.utils.data import Dataset
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
from matcho import Unet2D
# from YourDataset import YourDataset  # Import your custom dataset here
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
from torchinfo import summary
import torchprofile

import pickle

torch.manual_seed(23)

scaler = GradScaler()

DTYPE = torch.float32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 200
plt.rcParams["font.family"] = "serif"

import scipy.stats as stats

Using device: cuda


In [2]:
# Define your custom loss function here
class CustomLoss(nn.Module):
    def __init__(self, Par):
        super(CustomLoss, self).__init__()
        self.Par = Par

    def forward(self, y_pred, y_true):
        # Implement your custom loss calculation here
        # loss = torch.mean((y_pred - y_true) ** 2)  # Example: Mean Squared Error
        y_true = (y_true - self.Par["out_shift"])/self.Par["out_scale"]
        y_pred = (y_pred - self.Par["out_shift"])/self.Par["out_scale"]
        loss = torch.norm(y_true-y_pred, p=2)/torch.norm(y_true, p=2)
        return loss

class YourDataset_train(Dataset):
    def __init__(self, x, t, y, transform=None):
        self.x = x
        self.t = t
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x_sample = self.x[idx]
        t_sample = self.t[idx]
        y_sample = self.y[idx]

        if self.transform:
            x_sample, t_sample, y_sample = self.transform(x_sample, t_sample, y_sample)

        return x_sample, t_sample, y_sample
    
class YourDataset(Dataset):
    def __init__(self, x, y, transform=None):
        self.x = x
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x_sample = self.x[idx]
        y_sample = self.y[idx]

        if self.transform:
            x_sample, y_sample = self.transform(x_sample, y_sample)

        return x_sample, y_sample


# def preprocess(traj, Par):
#     x = sliding_window_view(traj[:,:-(Par['lf']),:,:,:,:], window_shape=Par['lb'], axis=1 ).transpose(0,1,6,2,3,4,5).reshape(-1,Par['lb'], Par['nf'], Par['nth'], Par['nr'], Par['nx'])
#     y = sliding_window_view(traj[:,Par['lb']:,:,:,:,:], window_shape=Par['lf'], axis=1 ).transpose(0,1,6,2,3,4,5).reshape(-1,Par['lf'], Par['nf'], Par['nth'], Par['nr'], Par['nx'])
#     t = np.linspace(0,1,Par['lf']).reshape(-1,1)

#     nt = y.shape[1]
#     n_samples = y.shape[0]

#     t = np.tile(t, [n_samples,1]).reshape(-1,)                                                     #[_*nt, ]
#     x = np.repeat(x,nt, axis=0)                                   #[_*nt, 1, 64, 64]
#     y = y.reshape(y.shape[0]*y.shape[1],1,y.shape[2],y.shape[3])  #[_*nt, 64, 64]


#     print('x: ', x.shape)
#     print('y: ', y.shape)
#     print('t: ', t.shape)
#     print()
#     return x,y,t

def preprocess_train(traj, Par):
    nsamples = traj.shape[0]
    nt = traj.shape[1]
    temp = nt - Par['lb'] - Par['lf'] + 1
    x_idx = np.arange(temp).reshape(-1,1)
    x_idx = np.tile(x_idx, (1, Par['lf'])).reshape(-1,1)

    x_idx_ls = []
    for i in range(Par["lb"]):
        x_idx_ls.append(x_idx+i)
    x_idx = np.concatenate(x_idx_ls, axis=1)

    t_idx = np.arange(Par['lf']).reshape(1,-1)

    t_idx = np.tile(t_idx, (temp,1)).reshape(-1,)

    y_idx = np.arange(nt)
    y_idx = sliding_window_view(y_idx[Par['lb']:], window_shape=Par['lf']).reshape(-1,)

    print(f"x_idx: {x_idx.shape}")
    print(f"t_idx: {t_idx.shape}")
    print(f"y_idx: {y_idx.shape}")

    return x_idx, t_idx, y_idx

def preprocess(traj, Par):
    nsamples = traj.shape[0]
    nt = traj.shape[1]
    temp = nt - Par['lb'] - Par['LF'] + 1
    x_idx = np.arange(temp).reshape(-1,1)
    # x_idx = np.tile(x_idx, (1, Par['LF'])).reshape(-1,1)

    x_idx_ls = []
    for i in range(Par["lb"]):
        x_idx_ls.append(x_idx+i)
    x_idx = np.concatenate(x_idx_ls, axis=1)

    t_idx = np.arange(Par['lf']).reshape(-1,)
    # t_idx = np.tile(t_idx, (temp,1)).reshape(-1,)

    y_idx = np.arange(nt)
    y_idx = sliding_window_view(y_idx[Par['lb']:], window_shape=Par['LF'])#.reshape(-1,)

    print(f"x_idx: {x_idx.shape}")
    print(f"t_idx: {t_idx.shape}")
    print(f"y_idx: {y_idx.shape}")

    return x_idx, t_idx, y_idx


def combined_scheduler(optimizer, total_epochs, warmup_epochs, last_epoch=-1):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / warmup_epochs
        else:
            return 0.5 * (1 + math.cos(math.pi * (epoch - warmup_epochs) / (total_epochs - warmup_epochs)))

    return LambdaLR(optimizer, lr_lambda, last_epoch)


def rollout(model, x,t,NT, Par, batch_size):
    # x - [bs, lb, nf, nx,ny]
    # t - [lf,]
    # NT - length of target time-series

    # print('NT: ', NT)

    y_pred_ls = []
    # lf = 
    # nx = 64
    # ny = 64

    bs = batch_size
    end= bs
    # for end in range(bs, x.shape[0]+1, bs):
    while True:
        start = end-bs
        out_ls = []
        
        temp_x1 = x[start:end] #[BS, lb, nf, nx,ny]
        out_ls = [temp_x1.to(device)]
        traj = torch.cat(out_ls, dim=1)

        while traj.shape[1] < NT:
            # model.eval()
            with torch.no_grad():
                temp_x = torch.repeat_interleave(temp_x1, Par['lf'], dim=0) #[BS*lf, lb, nf, nx,ny]
                temp_t = t.repeat(traj.shape[0]) #[BS*lf, ]
                with autocast():
                    out = model(temp_x.to(device), temp_t.to(device)).reshape(-1,Par['lf'], Par['nf'],Par['nx'],Par['ny']) #[BS, lf, nf, nx,ny]
                # print('out: ', out.shape)
                out_ls.append(out)
                traj = torch.cat(out_ls, dim=1)
                temp_x1 = traj[:,-Par['lb']:] #[BS, lb, nf, nx,ny]
                
        pred = torch.cat(out_ls, dim=1)[:, Par['lb']:NT] #[BS, lf, nf, nx, ny]
        # print('pred: ', pred.shape)
        y_pred_ls.append(pred)

        end = end+bs
        if end-bs > x.shape[0]+1:
            break
    
    # print("hello")
    # print(y_pred_ls)
    y_pred = torch.cat(y_pred_ls, dim=0)#.reshape(1,-1,Par['nz'],Par['ny'],Par['nx'])
    # print('y_pred: ', y_pred.shape)

    return y_pred

In [3]:
# Load your data into NumPy arrays (x_train, t_train, y_train, x_val, t_val, y_val, x_test, t_test, y_test)
#########################
res = 128
begin_time = time.time()
traj = np.load(f"../data/UX_nan_filtered.npy") #[nt, nx, ny]
traj = np.expand_dims(traj, axis=0) #[1, nt, nx, ny]
mask = np.load("../data/mask.npy").reshape(1,1,traj.shape[-2], traj.shape[-1])
traj = traj * mask
traj = np.expand_dims(traj, axis=2) #[1, nt, nf, nx, ny]
print(f"traj: {traj.shape}")
print(f"Data Loading Time: {time.time() - begin_time:.1f}s")


traj_train = traj[:, :800]
traj_val   = traj[:, 800:900,]
traj_test  = traj[:, 900:]

Par = {}
# Par['nt'] = 100 
Par['nx'] = traj_train.shape[-2]
Par['ny'] = traj_train.shape[-1]
Par['nf'] = 1
Par['d_emb'] = 128

Par['lb'] = 10
Par['lf'] = 2
Par['LF'] = 10
Par['channels'] = Par['nf']*Par['lb']
# Par['temp'] = Par['nt'] - Par['lb'] - Par['lf'] + 2

Par['num_epochs'] = 500 #50

time_cond = np.linspace(0, 1, Par['lf'])
if Par['lf']==1:
    time_cond = np.linspace(0, 1, Par['lf']) + 1


begin_time = time.time()
print('\nTrain Dataset')
x_idx_train, t_idx_train, y_idx_train = preprocess(traj_train, Par)
print('\nValidation Dataset')
x_idx_val, t_idx_val, y_idx_val  = preprocess(traj_val, Par)
print('\nTest Dataset')
x_idx_test, t_idx_test, y_idx_test  = preprocess(traj_test, Par)
print(f"Data Preprocess Time: {time.time() - begin_time:.1f}s")

# sys.exit()

t_min = np.min(time_cond)
t_max = np.max(time_cond)
if Par['lf']==1:
    t_min=0
    t_max=1


Par['inp_shift'] = np.mean(traj_train) 
Par['inp_scale'] = np.std(traj_train)
Par['out_shift'] = np.mean(traj_train)
Par['out_scale'] = np.std(traj_train)
Par['t_shift']   = t_min
Par['t_scale']   = t_max - t_min


with open('Par.pkl', 'wb') as f:
    pickle.dump(Par, f)

# sys.exit()
#########################

# Create custom datasets
mask_tensor = torch.tensor(mask, dtype=DTYPE, device=device)

Par["mask"] = mask_tensor

# Create custom datasets
traj_train_tensor = torch.tensor(traj_train, dtype=DTYPE)
traj_val_tensor = torch.tensor(traj_val, dtype=DTYPE)
traj_test_tensor = torch.tensor(traj_test, dtype=DTYPE)
time_cond_tensor = torch.tensor(time_cond, dtype=DTYPE)


train_dataset = YourDataset(x_idx_train, y_idx_train)
val_dataset = YourDataset(x_idx_val, y_idx_val)
test_dataset = YourDataset(x_idx_test, y_idx_test)


# Define data loaders
train_batch_size = 20 #100
val_batch_size   = 20 #100
test_batch_size  = 20 #100
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=val_batch_size)
test_loader = DataLoader(test_dataset, batch_size=test_batch_size)

traj: (1, 1000, 1, 128, 512)
Data Loading Time: 0.3s

Train Dataset
x_idx: (781, 10)
t_idx: (2,)
y_idx: (781, 10)

Validation Dataset
x_idx: (81, 10)
t_idx: (2,)
y_idx: (81, 10)

Test Dataset
x_idx: (81, 10)
t_idx: (2,)
y_idx: (81, 10)
Data Preprocess Time: 0.0s


In [4]:
model = Unet2D(dim=16, Par=Par, dim_mults=(1, 2, 4, 8), channels=Par['channels']).to(device).to(torch.float32)

path_model = 'models/best_model.pt'
model.load_state_dict(torch.load(path_model))

dummy_x = torch.tensor(np.random.uniform(size=(1,Par['lb'], Par['nf'], Par['nx'], Par['ny'])), dtype=DTYPE )
dummy_t = torch.tensor(np.random.uniform(size=(1,)), dtype=DTYPE )
dummy_input = (dummy_x.to(device), dummy_t.to(device))

print(summary(model, input_size=(dummy_x.shape , dummy_t.shape) ) )


# # Adjust the dimensions as per your model's input size
# dummy_x = traj_train_tensor[0, 0:1, ].to(device)
# dummy_t = time_cond_tensor[0:1].to(device)
# dummy_input = (dummy_x, dummy_t)

# # Profile the model
flops = torchprofile.profile_macs(model, dummy_input)
print(f"FLOPs: {flops:.2e}")

# Define loss function and optimizer
criterion = CustomLoss(Par)

Layer (type:depth-idx)                                  Output Shape              Param #
Unet2D                                                  [1, 1, 128, 512]          --
├─Conv2d: 1-1                                           [1, 16, 128, 512]         7,856
├─Sequential: 1-2                                       [1, 64]                   --
│    └─SinusoidalPosEmb: 2-1                            [1, 16]                   --
│    └─Linear: 2-2                                      [1, 64]                   1,088
│    └─GELU: 2-3                                        [1, 64]                   --
│    └─Linear: 2-4                                      [1, 64]                   4,160
├─ModuleList: 1-3                                       --                        --
│    └─ModuleList: 2-5                                  --                        --
│    │    └─ResnetBlock: 3-1                            [1, 16, 128, 512]         6,784
│    │    └─ResnetBlock: 3-2                    

/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::reshape". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::arange". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::exp". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::unsqueeze". Skipped.
  warnings.warn('No handlers found: "{}". Skipped.'.format(
/oscar/home/voommen/apps/torch_env/lib64/python3.9/site-packages/torchprofile/profile.py:22: UserWarning: No handlers found: "aten::sin". Skipped.
  warnings.w

# Sanity Check

In [10]:
y_true_ls = []
y_pred_ls = []

model.eval()
train_loss = 0.0
with torch.no_grad():
    for x_idx, y_idx in train_loader:
        x = traj_train_tensor[0, x_idx]        #[BS, lb, nf, nx, ny]
        t = time_cond_tensor[t_idx_train]      #[lf, ]
        y_true = traj_train_tensor[0, y_idx]   #[BS,lf, nf, nx, ny]
        y_pred = rollout(model, x,t,Par['lb']+Par['LF'], Par, train_batch_size)
        # print(f"y_true: {y_true.shape}")
        # print(f"y_pred: {y_pred.shape}")
        with autocast():
            loss   = criterion(y_pred, y_true.to(device))
        train_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

train_loss /= len(train_loader)
print(f"Train Loss: {train_loss:.4e}")

TRAIN_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['LF'], Par['nx'], Par['ny']).astype(np.float32)
TRAIN_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['LF'], Par['nx'], Par['ny']).astype(np.float32)

print(f"TRAIN_TRUE: {TRAIN_TRUE.shape}, DTYPE: {TRAIN_TRUE.dtype}")
print(f"TRAIN_PRED: {TRAIN_PRED.shape}, DTYPE: {TRAIN_PRED.dtype}")



y_true_ls = []
y_pred_ls = []

model.eval()
val_loss = 0.0
with torch.no_grad():
    for x_idx, y_idx in val_loader:
        x = traj_val_tensor[0, x_idx]        #[BS, lb, nf, nx, ny]
        t = time_cond_tensor[t_idx_val]      #[lf, ]
        y_true = traj_val_tensor[0, y_idx]   #[BS,lf, nf, nx, ny]
        y_pred = rollout(model, x,t,Par['lb']+Par['LF'], Par, val_batch_size)
        # print(f"y_true: {y_true.shape}")
        # print(f"y_pred: {y_pred.shape}")
        with autocast():
            loss   = criterion(y_pred, y_true.to(device))
        val_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

val_loss /= len(val_loader)
print(f"Val Loss: {val_loss:.4e}")

VAL_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['LF'], Par['nx'], Par['ny']).astype(np.float32)
VAL_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['LF'], Par['nx'], Par['ny']).astype(np.float32)

print(f"VAL_TRUE: {VAL_TRUE.shape}, DTYPE: {VAL_TRUE.dtype}")
print(f"VAL_PRED: {VAL_PRED.shape}, DTYPE: {VAL_PRED.dtype}")



y_true_ls = []
y_pred_ls = []

model.eval()
test_loss = 0.0
with torch.no_grad():
    for x_idx, y_idx in test_loader:
        x = traj_test_tensor[0, x_idx]        #[BS, lb, nf, nx, ny]
        t = time_cond_tensor[t_idx_test]      #[lf, ]
        y_true = traj_test_tensor[0, y_idx]   #[BS,lf, nf, nx, ny]
        y_pred = rollout(model, x,t,Par['lb']+Par['LF'], Par, test_batch_size)
        # print(f"y_true: {y_true.shape}")
        # print(f"y_pred: {y_pred.shape}")
        with autocast():
            loss   = criterion(y_pred, y_true.to(device))
        test_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

test_loss /= len(test_loader)
print(f"Test Loss: {test_loss:.4e}")

TEST_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['LF'], Par['nx'], Par['ny']).astype(np.float32)
TEST_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['LF'], Par['nx'], Par['ny']).astype(np.float32)

print(f"TEST_TRUE: {TEST_TRUE.shape}, DTYPE: {TEST_TRUE.dtype}")
print(f"TEST_PRED: {TEST_PRED.shape}, DTYPE: {TEST_PRED.dtype}")

Train Loss: 1.6036e-01
TRAIN_TRUE: (781, 10, 128, 512), DTYPE: float32
TRAIN_PRED: (781, 10, 128, 512), DTYPE: float32
Val Loss: 1.8452e-01
VAL_TRUE: (81, 10, 128, 512), DTYPE: float32
VAL_PRED: (81, 10, 128, 512), DTYPE: float32
Test Loss: 1.8576e-01
TEST_TRUE: (81, 10, 128, 512), DTYPE: float32
TEST_PRED: (81, 10, 128, 512), DTYPE: float32


In [11]:
np.save("TRAIN_TRUE.npy", TRAIN_TRUE)
np.save("TRAIN_PRED.npy", TRAIN_PRED)

np.save("VAL_TRUE.npy", VAL_TRUE)
np.save("VAL_PRED.npy", VAL_PRED)

np.save("TEST_TRUE.npy", TEST_TRUE)
np.save("TEST_PRED.npy", TEST_PRED)

In [12]:
sample1 = TRAIN_TRUE[1]
sample2 = TRAIN_TRUE[51]

err = np.abs(sample1 - sample2)
print(f"max err: {np.max(err)}")
print(f"min err: {np.min(err)}")

max err: 1.0577046871185303
min err: 0.0


In [17]:
y_true_ls = []
y_pred_ls = []

model.eval()
train_loss = 0.0
with torch.no_grad():
    for x, t, _, y_true in train_loader:
        with autocast():
            y_pred = model(x.to(device), t.to(device))
            loss   = criterion(y_pred, y_true.to(device))
        train_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

train_loss /= len(train_loader)
print(f"Train Loss: {train_loss:.4e}")

TRAIN_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
TRAIN_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"TRAIN_TRUE: {TRAIN_TRUE.shape}, DTYPE: {TRAIN_TRUE.dtype}")
print(f"TRAIN_PRED: {TRAIN_PRED.shape}, DTYPE: {TRAIN_PRED.dtype}")



y_true_ls = []
y_pred_ls = []

model.eval()
val_loss = 0.0
with torch.no_grad():
    for x, t, _, y_true in val_loader:
        with autocast():
            y_pred = model(x.to(device), t.to(device))
            loss   = criterion(y_pred, y_true.to(device))
        val_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

val_loss /= len(val_loader)
print(f"Val Loss: {val_loss:.4e}")

VAL_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
VAL_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"VAL_TRUE: {VAL_TRUE.shape}, DTYPE: {VAL_TRUE.dtype}")
print(f"VAL_PRED: {VAL_PRED.shape}, DTYPE: {VAL_PRED.dtype}")



y_true_ls = []
y_pred_ls = []

model.eval()
test_loss = 0.0
with torch.no_grad():
    for x, t, _, y_true in test_loader:
        with autocast():
            y_pred = model(x.to(device), t.to(device))
            loss   = criterion(y_pred, y_true.to(device))
        test_loss += loss.item()
        y_true_ls.append(y_true.detach().cpu().numpy())
        y_pred_ls.append(y_pred.detach().cpu().numpy())

test_loss /= len(test_loader)
print(f"Test Loss: {test_loss:.4e}")

TEST_TRUE = np.concatenate(y_true_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)
TEST_PRED = np.concatenate(y_pred_ls, axis=0).reshape(-1, Par['lf'], Par['nx'], Par['ny']).astype(np.float32)

print(f"TEST_TRUE: {TEST_TRUE.shape}, DTYPE: {TEST_TRUE.dtype}")
print(f"TEST_PRED: {TEST_PRED.shape}, DTYPE: {TEST_PRED.dtype}")

Train Loss: 2.9746e-01
TRAIN_TRUE: (800, 48, 128, 128), DTYPE: float32
TRAIN_PRED: (800, 48, 128, 128), DTYPE: float32
Val Loss: 4.2980e-01
VAL_TRUE: (100, 48, 128, 128), DTYPE: float32
VAL_PRED: (100, 48, 128, 128), DTYPE: float32
Test Loss: 4.4600e-01
TEST_TRUE: (100, 48, 128, 128), DTYPE: float32
TEST_PRED: (100, 48, 128, 128), DTYPE: float32


In [18]:
np.save("TRAIN_TRUE.npy", TRAIN_TRUE)
np.save("TRAIN_PRED.npy", TRAIN_PRED)

np.save("VAL_TRUE.npy", VAL_TRUE)
np.save("VAL_PRED.npy", VAL_PRED)

np.save("TEST_TRUE.npy", TEST_TRUE)
np.save("TEST_PRED.npy", TEST_PRED)

In [13]:
sample1 = TRAIN_TRUE[1]
sample2 = TRAIN_TRUE[51]

err = np.abs(sample1 - sample2)
print(f"max err: {np.max(err)}")
print(f"min err: {np.min(err)}")

max err: 6.0196990966796875
min err: 1.9073486328125e-06
